# Obtaining Honest Population-Level Inference from a Complex Survey

Almost no real-world social survey is simple random sampling. Polls, health surveys, and household panels are designed with three fundamental features: **sampling weights** (how many people in the target population each respondent represents), **stratification** (partitioning the population by region or age before sampling within strata), and **primary sampling units (PSU)** (clusters are sampled first—e.g., villages or neighborhoods—then individuals within them). If you ignore this design and treat the sample as independent and identically distributed, running OLS directly, you hit two traps simultaneously: point estimates become **biased** (unweighted estimates describe the "sample," not the "target population"), and standard errors become **falsely small** (ignoring within-PSU correlation overstates precision, leading to intervals that are too narrow and spurious *p*-values).

The remedy is called **design-based inference**: explicitly declare the survey design, let weights enter the point estimate and PSU enter the variance formula, yielding estimates that target the **population** and give honest intervals. This is what R's `survey` package and Python's `samplics` do—`svydesign()` declares the design, `svyglm()` fits weighted regression, and variance uses either Taylor linearization or resampling. Beyond estimation, rigorous surveys must answer two measurement questions **before data collection**: Is the scale reliable (Cronbach's α for internal consistency)? Is the sample large enough (power and sample size, accounting for clustering via the design effect DEFF, which shrinks effective *n*)?  

This tutorial follows the workflow step by step: load data → declare sampling design → pre-collection measurement and power design → design-weighted regression → weighted versus naive comparison → visualization → summary. We use `socialverse` to complete the entire pipeline. It wraps these methods thinly atop `statsmodels` and NumPy, using standard social-science conventions, and automatically creates an auditable provenance trail at each step—you'll see this at the end.

The data is a built-in synthetic survey: 300 respondents, 6 Likert items (item1...item6, driven by a single latent variable so reliability is naturally high), plus design columns `weight`, `strata`, and `psu`; a binary exposure and binary outcome.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import socialverse as sv
from socialverse import datasets as ds

## Load Survey Data

Read the table into memory and inspect it. Each row is a respondent: the first 6 columns are scale items (scored 1–5), `weight` is the sampling weight, `strata` is the stratum identifier, `psu` is the primary sampling unit ID, and `exposure` and `outcome` are the exposure and outcome of interest.

In [2]:
df = ds.load_survey()
print("调查维度:", df.shape)  # (300, 11)
df.head()

调查维度: (300, 11)


,item1,item2,item3,item4,item5,item6,weight,strata,psu,exposure,outcome
0,4,2,3,2,4,3,2.002,2,16,1,0
1,3,2,4,3,3,2,2.260,1,11,0,1
2,4,4,4,4,3,5,2.974,2,13,1,1
3,1,4,2,3,3,2,1.894,2,14,1,1
4,3,3,3,2,2,2,1.602,1,6,1,1


## Declare Target Estimands and Survey Design

Survey analysis starts not with data, but with **what you want to estimate**. We want to estimate the **population proportion / prevalence** of the binary outcome in the target population and its association with exposure. Encode this target in the study state; all downstream measurement design and estimation read from it.

In [3]:
st = sv.StudyState()
st.write("estimand", "target", "prevalence")  # 目标量:面向总体的占比/关联,而非样本描述

sv.pp.ingest(st, data=df)  # 把这份调查表登记进研究状态
st.sources["dataset_meta"]

{'kind': 'DataFrame',
 'shape': (300, 11),
 'columns': ['item1',
  'item2',
  'item3',
  'item4',
  'item5',
  'item6',
  'weight',
  'strata',
  'psu',
  'exposure',
  'outcome']}

Next comes the **watershed between complex survey design and "just run a regression"**: explicitly declare three design facts. `weights='weight'` ensures estimates target the population (adjusting for unequal inclusion probabilities); `strata='strata'` records stratification so variance is computed within strata; `psu='psu'` identifies the cluster unit—observations within the same PSU are correlated, so standard errors must be clustered accordingly, otherwise intervals are spuriously narrow. These three column names are stored in the study state, and downstream weighted regression reads them to do its work.

In [4]:
sv.pp.declare_design(
    st,
    weights="weight",   # 抽样权重列 → 点估计面向总体
    strata="strata",    # 分层列 → 方差按层
    psu="psu",          # 初级抽样单元(聚类)列 → 标准误按 PSU 聚类
)
dict(st.design)

{'weights': 'weight', 'strata': 'strata', 'psu': 'psu'}

## Pre-Collection: Scale Reliability and Sample Size

Before estimation, ask two questions: **Is the scale reliable?** (Do the 6 items consistently measure one construct?) **Is the sample large enough?** (What sample size is needed to achieve target power, inflated for clustering design via DEFF?). `design_survey` computes both in one step: Cronbach's α from the item matrix, and required sample size using normal approximation, adjusted upward by the design effect DEFF.

Cronbach's α is the internal consistency reliability coefficient, ranging 0–1; higher values indicate items are measuring the same latent construct. An α ≥ 0.9 is considered "excellent" by convention. Sample size is computed as *n* ≈ ((*z*_{α/2} + *z*_β) / effect size)² × DEFF: DEFF (design effect) exceeds 1 because clustering reduces effective sample size, so multiply the nominal *n* by DEFF to get the true required size.

In [5]:
items = df[[c for c in df.columns if c.startswith("item")]]  # 6 道 Likert 题项作响应矩阵
sv.tl.design_survey(
    st,
    items=items,
    effect_size=0.2,  # 标准化效应(小效应)
    deff=1.5,         # 设计效应:聚类使方差膨胀 50%
    alpha=0.05,
    power=0.8,
)

rel = st.diagnostics["reliability"]
pow_ = st.diagnostics["power"]
print(f"Cronbach α = {rel['alpha']:.3f}   (k = {rel['k']} 题, n = {rel['n_respondents']} 人)")
print(f"所需样本量 n = {pow_['n_required']}   (已按 DEFF = {pow_['deff']} 膨胀, 每组 {pow_['n_per_group']})")

Cronbach α = 0.932   (k = 6 题, n = 300 人)
所需样本量 n = 295   (已按 DEFF = 1.5 膨胀, 每组 148)


Reading: **α ≈ 0.932**—the 6 items show excellent internal consistency; the scale is reliable. Power calculation shows **required *n* ≈ 295** (inflated by DEFF = 1.5), and we have exactly 300, meeting the threshold barely. This is the most overlooked lesson in clustered survey design: 300 respondents sounds ample until you account for clustering, at which point effective information shrinks to just adequate. If you reset `deff` to 1.0 (pretending simple random sampling), required *n* becomes noticeably smaller—**ignoring clustering fools you into thinking the sample suffices**.

## Design-Weighted Regression: The Heart of the Chain

Now conduct post-collection estimation. This is not ordinary OLS; three things happen simultaneously. First, **weighting**: use declared sampling weights in WLS to target the population. Second, **cluster-robust variance**: because PSU is declared, `statsmodels` uses `cov_type="cluster"` to cluster standard errors by PSU, yielding honest (usually wider) intervals. Third, **naive comparison**: fit unweighted OLS in parallel, storing both weighted and unweighted coefficients side by side to assess whether weights materially shift the estimate.

Estimation requires declaring the outcome variable first, so we add the binary outcome `outcome` to the study state, then pass the exposure `exposure` as the predictor, fitting `outcome ~ exposure`.

In [6]:
st.write("variables", "outcome", "outcome")  # 声明二元结局列

sv.tl.survey_estimate(st, exposure="exposure")  # WLS + PSU 聚类稳健:outcome ~ exposure

m = st.models["weighted_reg"]
print(f"exposure 系数 = {m['coef']['exposure']:.3f}")
print(f"95% CI        = [{m['ci']['exposure'][0]:.3f}, {m['ci']['exposure'][1]:.3f}]")
print(f"SE            = {m['se']['exposure']:.3f}")
print(f"方差类型      = {m['cov_type']}   (n = {m['n']})")

exposure 系数 = 0.205
95% CI        = [0.116, 0.295]
SE            = 0.046
方差类型      = cluster   (n = 300)


Reading: the design-weighted coefficient on `exposure` is ≈ **0.205**, 95% CI ≈ **[0.116, 0.295]**; the interval excludes zero, so exposure and outcome are positively associated. The crucial detail is the `cov_type` field—it reads **cluster**, confirming this interval was computed with PSU clustering, not naive iid assumptions. That single field is itself evidence: it documents what makes this interval honest.

## Weighted vs. Naive: Design Declaration Is Not Merely Ceremonial

How do you verify that declaring the design actually matters? Compare the design-weighted coefficient to the naive (unweighted) OLS coefficient side by side. If they differ substantially, weights have shifted the conclusion. If they are similar, the value lies chiefly in **variance**—cluster-robust standard errors are typically wider than iid, giving more honest intervals. This comparison table is your direct answer to a reviewer's inevitable question: "Did you account for the survey design?" It goes straight into your supplementary materials.

In [7]:
import pandas as pd

sens = st.diagnostics["sensitivity"]
pd.DataFrame({
    "设计加权": sens["weighted"],
    "朴素 OLS": sens["unweighted"],
    "差异 delta": sens["delta"],
})

,设计加权,朴素 OLS,差异 delta
const,0.458246,0.447368,0.010878
exposure,0.205097,0.214794,-0.009697


In this gently weighted dataset, the design-weighted coefficient on `exposure` (0.205) and naive coefficient (0.215) differ by roughly 0.01—weights do not dramatically shift point estimates. This is normal; this dataset's weights are not extreme. The real gain is on the variance side: cluster-robust standard errors widen the interval and make it more credible. `survey_estimate` stores a clean coefficient table in the study state, with both weighted and unweighted columns, ready for a journal appendix.

In [8]:
st.artifacts["tables"]

,term,coef,se,ci_low,ci_high,coef_unweighted
0,const,0.458246,0.031175,0.397144,0.519348,0.447368
1,exposure,0.205097,0.045650,0.115624,0.294570,0.214794


## Visualization: Plot the Weighted Estimates

Deliverables. `survey_dist` reads the regression results from the study state and plots a forest-plot-style horizontal bar chart: each non-intercept term gets one bar showing the **design-weighted** point estimate plus a 95% CI whisker, with the **naive (unweighted)** estimate overlaid as a light-colored diamond, so readers see at a glance how much weights move each coefficient. The plot saves to a relative path in the notebook directory.

In [9]:
sv.pl.survey_dist(st, out="fig_survey.png", title="调查加权估计 · outcome ~ exposure")
print("图已保存:fig_survey.png")

图已保存:fig_survey.png


The blue bar is the design-weighted point estimate and cluster-robust 95% CI; the gray diamond is the naive unweighted comparison. This plot, together with the coefficient table above, comprises the deliverables from this chain:

![Design-weighted coefficient plot](fig_survey.png)

## Reproducible Provenance Chain

Finally, observe the key difference between `socialverse` and a typical estimation script. Running the entire chain, the study state automatically accumulates a **provenance ledger**: which function was called at each step, what it consumed, what it produced. This ledger is not a post-hoc log you write afterward; it is a byproduct of every call. When a reviewer asks "How did you do it? Did you account for weights and clustering?" the answer is already written incrementally here. The same contract also guards against misuse: if you attempt weighted estimation without declaring the design, the registry throws an error immediately, telling you what precondition is missing, rather than silently delivering a wrong interval.

In [10]:
print(st.summary())

StudyState
  sources: datasets, dataset_name, dataset_meta
  design: weights, strata, psu, sampling_frame
  variables: scales, constructs, outcome
  estimand: target
  models: weighted_reg
  diagnostics: reliability, power, sensitivity
  artifacts: tables, figures
  provenance: 5 step(s)


## Summary

We traced a complete design-based weighted inference pipeline: declare design (weights/strata/PSU) → measure reliability α and compute power/sample size → WLS + PSU cluster-robust weighted regression → weighted versus naive sensitivity → visualization. Methodologically this is equivalent to **R's `survey` package** (Lumley) `svydesign()` + `svyglm()`, **Python `samplics`**, and measurement tools comparable to `psych::alpha` and `pwr`. socialverse does not reimplement these methods; it wraps them thinly over `statsmodels.WLS(cov_type="cluster")` and pure NumPy formulas for α and power.

Relative to stand-alone estimation tools, this approach adds two things. First, scale reliability and the cluster-adjusted sample size are **laid out before data collection**, not discovered after analysis reveals the sample was insufficient. Second, an auditable provenance chain threads through the entire analysis, and attempting estimation without declaring design triggers a hard error. The next tutorial, [04_econometrics_replication](04_econometrics_replication.ipynb), shifts to econometric replication workflows.